In [0]:
# Create environment parameter widget
# Valid values: dev, uat, prod
dbutils.widgets.text("environment", "dev")

# Get the environment value
environment = dbutils.widgets.get("environment")
print(f"Environment: {environment}")

In [0]:
# Configuration: Map environment to catalog name
# Using underscores instead of hyphens for better SQL compatibility
CATALOG_CONFIG = {
    'dev': 'AdventureWorksDW_dev',
    'uat': 'AdventureWorksDW_uat',
    'prod': 'AdventureWorksDW_prod'
}

# Validate environment
environment = dbutils.widgets.get("environment")
if environment not in CATALOG_CONFIG:
    raise ValueError(f"Invalid environment: '{environment}'. Valid values: {list(CATALOG_CONFIG.keys())}")

catalog_name = CATALOG_CONFIG[environment]

try:
    # Create catalog if it doesn't exist
    spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS `{catalog_name}`
        COMMENT 'AdventureWorks Data Warehouse - {environment.upper()} environment'
    """)
    
    # Set this catalog as the default context
    spark.sql(f"USE CATALOG `{catalog_name}`")
    
    print(f"✓ Catalog '{catalog_name}' is ready")
    print(f"✓ Active catalog set to: {catalog_name}")
    
except Exception as e:
    print(f"✗ Error creating catalog: {e}")
    raise

In [0]:
# Schema configuration: medallion architecture layers
# Using underscores for consistency and SQL compatibility
SCHEMA_CONFIG = [
    {
        'name': 'bronze',
        'suffix': environment,
        'comment': 'Raw ingested data - minimal transformations'
    },
    {
        'name': 'silver',
        'suffix': environment,
        'comment': 'Cleaned and validated data - business logic applied'
    },
    {
        'name': 'gold',
        'suffix': environment,
        'comment': 'Aggregated business-level data - analytics-ready'
    }
]

# Create all schemas in the active catalog
print(f"Creating schemas in catalog: {catalog_name}\n")

created_schemas = []
errors = []

for schema_config in SCHEMA_CONFIG:
    schema_name = f"{schema_config['name']}_{schema_config['suffix']}"
    
    try:
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS `{schema_name}`
            COMMENT '{schema_config['comment']}'
        """)
        
        created_schemas.append(schema_name)
        print(f"✓ Schema '{schema_name}' created successfully")
        
    except Exception as e:
        error_msg = f"Schema '{schema_name}': {str(e)}"
        errors.append(error_msg)
        print(f"✗ {error_msg}")

# Summary
print(f"\n{'='*50}")
print(f"Summary: {len(created_schemas)}/{len(SCHEMA_CONFIG)} schemas ready")
if created_schemas:
    print(f"\nActive schemas:")
    for schema in created_schemas:
        print(f"  - {catalog_name}.{schema}")

if errors:
    print(f"\n⚠️ Errors encountered: {len(errors)}")
    raise Exception(f"Schema creation had {len(errors)} error(s)")

In [0]:
spark.sql("DROP CATALOG IF EXISTS adventureworksdw_dev CASCADE")
print("Success")